# Dynamic Test Prediction Collection

This notebook reproduces the published dynamic test-set results and exports aligned per-sample predictions for the image-sequence, FEA-sequence, and multimodal dynamic FER models.

It follows the same output semantics as `collect_static_test_predictions.ipynb`:

- `sample_id` identifies one concrete Central- or Side-view sample and is unique across the 756-row test set.
- `reenactment_id` identifies the underlying reenactment and therefore occurs twice, once per camera view.
- The dynamic FEA model produces one prediction per reenactment. That prediction is joined to both corresponding view-level rows for the final export.
- Dynamic Image and Multimodal predictions are produced separately for each of the 756 view-level sequences.

The current dynamic models already use the canonical EmoHeVRDB class order `Anger, Disgust, Fear, Happiness, Neutral, Sadness, Surprise`, so no class-ID remapping is required.

The image and multimodal inference pipelines load image sequences lazily and models are processed one at a time to keep CPU/GPU memory usage low.


## 1. Setup



### 1.0 Add log filter

Add the log filter below or tensorflow will print thousands of lines of uninformative log messages.  


In [1]:
import re
import ipykernel.iostream

TF_LOG_FILTER_PATTERNS = [
    r'ptx\d+.*is not a recognized feature for this target',
    r'is not a recognized feature for this target \(ignoring feature\)',
    r'\(ignoring feature\)',
    r'successful NUMA node read from SysFS had negative value \(-1\)',
    r'gpu_timer\.cc:114\] Skipping the delay kernel, measurement accuracy will be reduced',
]

KERAS_PROGRESS_PATTERNS = [
    r'ms/step',
    r's/step',
    r'ETA:',
    r'\d+/\d+ \[',   # 12/64 [===>...]
]

_original_write = ipykernel.iostream.OutStream.write

def _filtered_write(self, msg, *args, **kwargs):
    text = str(msg)

    if any(re.search(p, text) for p in KERAS_PROGRESS_PATTERNS):
        _original_write(self, text, *args, **kwargs)
        return

    buf = getattr(self, '_tf_log_filter_buf', '')
    buf += text

    if '\n' not in buf:
        setattr(self, '_tf_log_filter_buf', buf)
        return

    lines = buf.splitlines(keepends=True)
    if not buf.endswith('\n'):
        incomplete = lines.pop()
    else:
        incomplete = ''

    for line in lines:
        if any(re.search(p, line) for p in TF_LOG_FILTER_PATTERNS):
            continue
        _original_write(self, line, *args, **kwargs)

    setattr(self, '_tf_log_filter_buf', incomplete)

ipykernel.iostream.OutStream.write = _filtered_write

print('Notebook log filter installed (targeted, keeps Keras steps).')

Notebook log filter installed (targeted, keeps Keras steps).


### 1.1 Imports and paths

The paths below follow the `emohevrdb-dfer` Docker environment and repository conventions. If the downloaded model files are stored elsewhere, only adjust the three model paths here.


In [2]:
from pathlib import Path
import gc

import keras
import numpy as np
import pandas as pd
import tensorflow as tf


DATASET_ROOT = Path("/workspace/datasets")
MODEL_ROOT = Path("/workspace/emoji_hero_vr_dfer/models")

DI_TEST_PATH = DATASET_ROOT / "emoji-hero-vr-db-di" / "test_set"
DFEA_TEST_PATH = DATASET_ROOT / "emoji-hero-vr-db-dfea-as-csv" / "test_set.csv"

IMAGE_MODEL_PATH = MODEL_ROOT / "image_sequence_model.keras"
FEA_MODEL_PATH = MODEL_ROOT / "fea_sequence_model.keras"
MULTIMODAL_MODEL_PATH = MODEL_ROOT / "best_int_fusion_model.keras"

SEQUENCE_LENGTH = 30
IMAGE_SIZE = (224, 224, 3)
BATCH_SIZE = 32

keras.mixed_precision.set_global_policy("mixed_float16")

print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

for path in [DI_TEST_PATH, DFEA_TEST_PATH, IMAGE_MODEL_PATH, FEA_MODEL_PATH, MULTIMODAL_MODEL_PATH]:
    print(f"{path}: {'OK' if path.exists() else 'MISSING'}")


2026-09-17 17:17:58.340810: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-09-17 17:17:58.349226: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8473] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-09-17 17:17:58.351764: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1471] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


TensorFlow: 2.17.0
Keras: 3.12.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
/workspace/datasets/emoji-hero-vr-db-di/test_set: OK
/workspace/datasets/emoji-hero-vr-db-dfea-as-csv/test_set.csv: OK
/workspace/emoji_hero_vr_dfer/models/image_sequence_model.keras: OK
/workspace/emoji_hero_vr_dfer/models/fea_sequence_model.keras: OK
/workspace/emoji_hero_vr_dfer/models/best_int_fusion_model.keras: OK


### 1.2 Class mapping

All dynamic datasets and models use the current canonical EmoHeVRDB class order.


In [3]:
CANONICAL_ID_TO_EMOTION = {
    0: "Anger",
    1: "Disgust",
    2: "Fear",
    3: "Happiness",
    4: "Neutral",
    5: "Sadness",
    6: "Surprise"
}

CANONICAL_EMOTION_TO_ID = {v: k for k, v in CANONICAL_ID_TO_EMOTION.items()}
EMOTIONS = list(CANONICAL_EMOTION_TO_ID.keys())


## 2. Load and align the dynamic test data

### 2.1 Load the FEA-sequence test set

The dynamic FEA CSV contains 30 time-ordered observations per `sequence_id`. We use `reenactment_id` throughout this notebook for the same underlying identifier so that the exported dynamic predictions can later be aligned directly with the static predictions.


In [4]:
fea_frame_df = pd.read_csv(DFEA_TEST_PATH).rename(columns={"sequence_id": "reenactment_id"})

print(f"fea_frame_df shape: {fea_frame_df.shape}")
print(f"fea_frame_df columns: {fea_frame_df.columns.tolist()}")
display(fea_frame_df.head())

FEA_COLUMNS = [c for c in fea_frame_df.columns if c not in {"reenactment_id", "timestamp", "Label"}]

assert len(FEA_COLUMNS) == 63
assert fea_frame_df["reenactment_id"].nunique() == 378
assert (fea_frame_df.groupby("reenactment_id").size() == SEQUENCE_LENGTH).all()
assert fea_frame_df[FEA_COLUMNS].notna().all().all()


fea_frame_df shape: (11340, 66)
fea_frame_df columns: ['reenactment_id', 'timestamp', 'BrowLowererL', 'BrowLowererR', 'CheekPuffL', 'CheekPuffR', 'CheekRaiserL', 'CheekRaiserR', 'CheekSuckL', 'CheekSuckR', 'ChinRaiserB', 'ChinRaiserT', 'DimplerL', 'DimplerR', 'EyesClosedL', 'EyesClosedR', 'EyesLookDownL', 'EyesLookDownR', 'EyesLookLeftL', 'EyesLookLeftR', 'EyesLookRightL', 'EyesLookRightR', 'EyesLookUpL', 'EyesLookUpR', 'InnerBrowRaiserL', 'InnerBrowRaiserR', 'JawDrop', 'JawSidewaysLeft', 'JawSidewaysRight', 'JawThrust', 'LidTightenerL', 'LidTightenerR', 'LipCornerDepressorL', 'LipCornerDepressorR', 'LipCornerPullerL', 'LipCornerPullerR', 'LipFunnelerLB', 'LipFunnelerLT', 'LipFunnelerRB', 'LipFunnelerRT', 'LipPressorL', 'LipPressorR', 'LipPuckerL', 'LipPuckerR', 'LipStretcherL', 'LipStretcherR', 'LipSuckLB', 'LipSuckLT', 'LipSuckRB', 'LipSuckRT', 'LipTightenerL', 'LipTightenerR', 'LipsToward', 'LowerLipDepressorL', 'LowerLipDepressorR', 'MouthLeft', 'MouthRight', 'NoseWrinklerL', 'Nose

,reenactment_id,timestamp,BrowLowererL,BrowLowererR,CheekPuffL,CheekPuffR,CheekRaiserL,CheekRaiserR,CheekSuckL,CheekSuckR,...,MouthRight,NoseWrinklerL,NoseWrinklerR,OuterBrowRaiserL,OuterBrowRaiserR,UpperLidRaiserL,UpperLidRaiserR,UpperLipRaiserL,UpperLipRaiserR,Label
0,1700478995850-2-1-1-0-0,1700478994882,0.299859,0.315491,0.072744,0.060128,0.101609,0.059095,1.401298e-45,1.401298e-45,...,6.212495e-08,2.818527e-11,4.151673e-11,2.690706e-13,0.000217,1.401298e-45,1.401298e-45,0.015593,0.010760,0
1,1700478995850-2-1-1-0-0,1700478994922,0.274252,0.291602,0.084611,0.068393,0.098516,0.057481,1.401298e-45,1.401298e-45,...,4.365342e-08,1.985715e-11,2.925454e-11,2.224745e-13,0.000179,1.401298e-45,1.401298e-45,0.014098,0.010794,0
2,1700478995850-2-1-1-0-0,1700478994949,0.260239,0.291855,0.073218,0.067142,0.096496,0.056436,1.401298e-45,1.401298e-45,...,3.069010e-08,1.399077e-11,2.061487e-11,1.839484e-13,0.000148,1.401298e-45,1.401298e-45,0.022011,0.010817,0
3,1700478995850-2-1-1-0-0,1700478994977,0.266225,0.299843,0.074137,0.071606,0.095162,0.055750,1.401298e-45,1.401298e-45,...,2.158532e-08,9.857880e-12,1.452693e-11,1.520928e-13,0.000218,1.401298e-45,1.401298e-45,0.017628,0.010832,0
4,1700478995850-2-1-1-0-0,1700478995018,0.248875,0.273375,0.074750,0.068955,0.094270,0.055294,1.401298e-45,1.401298e-45,...,1.518726e-08,6.946256e-12,1.023726e-11,1.257550e-13,0.000180,1.401298e-45,1.401298e-45,0.015070,0.010843,0


In [5]:
fea_sequence_rows = []

for reenactment_id, group in fea_frame_df.groupby("reenactment_id", sort=True):
    group = group.sort_values("timestamp")

    assert group["Label"].nunique() == 1

    fea_sequence_rows.append({
        "reenactment_id": reenactment_id,
        "fea_timestamps": group["timestamp"].astype(np.int64).to_numpy(),
        "fea_sequence": group[FEA_COLUMNS].to_numpy(dtype=np.float32),
        "fea_true_label_id": int(group["Label"].iloc[0])
    })

fea_sequence_df = pd.DataFrame(fea_sequence_rows).sort_values("reenactment_id").reset_index(drop=True)

assert len(fea_sequence_df) == 378
assert fea_sequence_df["reenactment_id"].is_unique
assert all(sequence.shape == (SEQUENCE_LENGTH, 63) for sequence in fea_sequence_df["fea_sequence"])

display(fea_sequence_df.head())


,reenactment_id,fea_timestamps,fea_sequence,fea_true_label_id
0,1700478995850-2-1-1-0-0,"[1700478994882, 1700478994922, 1700478994949, ...","[[0.29985854, 0.31549126, 0.07274411, 0.060127...",0
1,1700478998549-2-1-1-1-5,"[1700478997506, 1700478997547, 1700478997576, ...","[[0.05101412, 0.018364852, 1.3082434e-12, 6.27...",5
2,1700479001137-2-1-1-2-3,"[1700479000163, 1700479000206, 1700479000234, ...","[[9.224355e-06, 9.613555e-06, 2.511383e-10, 2....",3
3,1700479004312-2-1-1-3-0,"[1700479003340, 1700479003381, 1700479003409, ...","[[0.09826975, 0.06595065, 0.010865643, 0.01086...",0
4,1700479005401-2-1-1-4-0,"[1700479004694, 1700479004731, 1700479004773, ...","[[0.26392713, 0.24405727, 0.010851177, 0.02469...",0


### 2.2 Parse the dynamic image-sequence test set

Each sequence directory name contains the reenactment identifier plus the camera index:

`<timestamp>-<set-id>-<participant-id>-<level-id>-<emoji-id>-<emotion-id>-<camera-index>`

The directory name is therefore also the view-level `sample_id`. Removing the final camera index yields the shared `reenactment_id`.


In [6]:
def parse_image_sequence_dir(sequence_dir: Path) -> dict:
    parts = sequence_dir.name.split("-")

    if len(parts) != 7:
        raise ValueError(f"Unexpected sequence directory name: {sequence_dir.name}")

    timestamp, set_id, participant_id, level_id, emoji_id, emotion_id, camera_index = parts
    reenactment_id = "-".join(parts[:-1])

    image_paths = sorted(
        [path for path in sequence_dir.iterdir() if path.is_file()],
        key=lambda path: int(path.name.split("-")[0])
    )
    frame_timestamps = np.array([int(path.name.split("-")[0]) for path in image_paths], dtype=np.int64)

    return {
        "sample_id": sequence_dir.name,
        "reenactment_id": reenactment_id,
        "timestamp": int(timestamp),
        "set_id": int(set_id),
        "participant_id": int(participant_id),
        "level_id": int(level_id),
        "emoji_id": int(emoji_id),
        "true_label_id": int(emotion_id),
        "camera_index": int(camera_index),
        "perspective": "Central" if camera_index == "0" else "Side",
        "image_sequence_path": str(sequence_dir),
        "image_paths": [str(path) for path in image_paths],
        "frame_timestamps": frame_timestamps,
        "emotion_dir": sequence_dir.parent.name
    }


image_rows = []

for class_dir in sorted(DI_TEST_PATH.iterdir()):
    if not class_dir.is_dir():
        continue

    for sequence_dir in sorted(class_dir.iterdir()):
        if sequence_dir.is_dir():
            image_rows.append(parse_image_sequence_dir(sequence_dir))

image_df = pd.DataFrame(image_rows).sort_values(["reenactment_id", "camera_index"]).reset_index(drop=True)

image_df.head()


,sample_id,reenactment_id,timestamp,set_id,participant_id,level_id,emoji_id,true_label_id,camera_index,perspective,image_sequence_path,image_paths,frame_timestamps,emotion_dir
0,1700478995850-2-1-1-0-0-0,1700478995850-2-1-1-0-0,1700478995850,2,1,1,0,0,0,Central,/workspace/datasets/emoji-hero-vr-db-di/test_s...,[/workspace/datasets/emoji-hero-vr-db-di/test_...,"[1700478994882, 1700478994922, 1700478994949, ...",Anger
1,1700478995850-2-1-1-0-0-1,1700478995850-2-1-1-0-0,1700478995850,2,1,1,0,0,1,Side,/workspace/datasets/emoji-hero-vr-db-di/test_s...,[/workspace/datasets/emoji-hero-vr-db-di/test_...,"[1700478994882, 1700478994922, 1700478994949, ...",Anger
2,1700478998549-2-1-1-1-5-0,1700478998549-2-1-1-1-5,1700478998549,2,1,1,1,5,0,Central,/workspace/datasets/emoji-hero-vr-db-di/test_s...,[/workspace/datasets/emoji-hero-vr-db-di/test_...,"[1700478997506, 1700478997547, 1700478997576, ...",Sadness
3,1700478998549-2-1-1-1-5-1,1700478998549-2-1-1-1-5,1700478998549,2,1,1,1,5,1,Side,/workspace/datasets/emoji-hero-vr-db-di/test_s...,[/workspace/datasets/emoji-hero-vr-db-di/test_...,"[1700478997506, 1700478997547, 1700478997576, ...",Sadness
4,1700479001137-2-1-1-2-3-0,1700479001137-2-1-1-2-3,1700479001137,2,1,1,2,3,0,Central,/workspace/datasets/emoji-hero-vr-db-di/test_s...,[/workspace/datasets/emoji-hero-vr-db-di/test_...,"[1700479000163, 1700479000206, 1700479000234, ...",Happiness


In [7]:
assert len(image_df) == 756
assert image_df["sample_id"].is_unique
assert image_df["reenactment_id"].nunique() == 378
assert set(image_df["camera_index"]) == {0, 1}

assert (image_df.groupby("reenactment_id").size() == 2).all()
assert (image_df.groupby("reenactment_id")["camera_index"].nunique() == 2).all()
assert image_df["image_paths"].map(len).eq(SEQUENCE_LENGTH).all()
assert image_df["frame_timestamps"].map(len).eq(SEQUENCE_LENGTH).all()

assert (image_df["emotion_dir"].map(CANONICAL_EMOTION_TO_ID) == image_df["true_label_id"]).all()
assert set(image_df["reenactment_id"]) == set(fea_sequence_df["reenactment_id"])

display(image_df.head())


,sample_id,reenactment_id,timestamp,set_id,participant_id,level_id,emoji_id,true_label_id,camera_index,perspective,image_sequence_path,image_paths,frame_timestamps,emotion_dir
0,1700478995850-2-1-1-0-0-0,1700478995850-2-1-1-0-0,1700478995850,2,1,1,0,0,0,Central,/workspace/datasets/emoji-hero-vr-db-di/test_s...,[/workspace/datasets/emoji-hero-vr-db-di/test_...,"[1700478994882, 1700478994922, 1700478994949, ...",Anger
1,1700478995850-2-1-1-0-0-1,1700478995850-2-1-1-0-0,1700478995850,2,1,1,0,0,1,Side,/workspace/datasets/emoji-hero-vr-db-di/test_s...,[/workspace/datasets/emoji-hero-vr-db-di/test_...,"[1700478994882, 1700478994922, 1700478994949, ...",Anger
2,1700478998549-2-1-1-1-5-0,1700478998549-2-1-1-1-5,1700478998549,2,1,1,1,5,0,Central,/workspace/datasets/emoji-hero-vr-db-di/test_s...,[/workspace/datasets/emoji-hero-vr-db-di/test_...,"[1700478997506, 1700478997547, 1700478997576, ...",Sadness
3,1700478998549-2-1-1-1-5-1,1700478998549-2-1-1-1-5,1700478998549,2,1,1,1,5,1,Side,/workspace/datasets/emoji-hero-vr-db-di/test_s...,[/workspace/datasets/emoji-hero-vr-db-di/test_...,"[1700478997506, 1700478997547, 1700478997576, ...",Sadness
4,1700479001137-2-1-1-2-3-0,1700479001137-2-1-1-2-3,1700479001137,2,1,1,2,3,0,Central,/workspace/datasets/emoji-hero-vr-db-di/test_s...,[/workspace/datasets/emoji-hero-vr-db-di/test_...,"[1700479000163, 1700479000206, 1700479000234, ...",Happiness


### 2.3 Verify labels, timestamps, and create the canonical prediction table

The FEA sequence for a reenactment is joined to both corresponding image-view sequences. The complementarity analysis in `emohevrdb-dfer` aligns the modalities frame-by-frame using timestamps; the same check is enforced here.


In [8]:
prediction_df = image_df.merge(
    fea_sequence_df[["reenactment_id", "fea_timestamps", "fea_sequence", "fea_true_label_id"]],
    on="reenactment_id",
    how="left",
    validate="many_to_one"
)

prediction_df["true_label"] = prediction_df["true_label_id"].map(CANONICAL_ID_TO_EMOTION)

assert len(prediction_df) == 756
assert prediction_df["sample_id"].is_unique
assert prediction_df["reenactment_id"].nunique() == 378
assert prediction_df["fea_sequence"].notna().all()

assert (prediction_df["true_label_id"] == prediction_df["fea_true_label_id"]).all()
assert all(np.array_equal(row.frame_timestamps, row.fea_timestamps) for row in prediction_df.itertuples())

assert (prediction_df.groupby("true_label_id").size() == 108).all()
assert (fea_sequence_df.groupby("fea_true_label_id").size() == 54).all()

print("Dynamic image and FEA sequences are aligned by reenactment, label, and all 30 timestamps.")


Dynamic image and FEA sequences are aligned by reenactment, label, and all 30 timestamps.


## 3. Inference utilities

### 3.1 Custom layers required for loading the frozen models

The dynamic image and multimodal models contain the custom `SequenceAugment` layer used during training. For inference this layer is a pass-through cast, which is exactly the behavior of the original layer when `training=False`.

The multimodal model additionally contains the custom cross-attention layer from the intermediate-fusion experiment.


In [9]:
from keras import layers
from keras.initializers import GlorotUniform
from keras.saving import register_keras_serializable


@register_keras_serializable(package="seqaug")
class SequenceAugment(layers.Layer):

    def __init__(self, image_size=(224, 224), crop_scale=(0.95, 1.0), rotation_max_deg=20.0,
                 fill_mode="CONSTANT", fill_value=1.0, flip_prob=0.5, brightness_max_delta=20.0,
                 contrast_lower=0.9, contrast_upper=1.1, gamma_range=(0.9, 1.1), clip_after_color=True,
                 noise_std=4, temporal_shift_max=0, frame_drop_prob=0.05, time_mask_prob=0.05,
                 time_mask_max_frac=0.10, invert_prob=0.0, solarize_prob=0.05,
                 solarize_threshold=(120.0, 160.0), **kwargs):
        
        super().__init__(**kwargs)
        self.image_size = tuple(image_size)
        self.crop_scale = tuple(crop_scale)
        self.rotation_max_deg = float(rotation_max_deg)
        self.fill_mode = str(fill_mode)
        self.fill_value = float(fill_value)
        self.flip_prob = float(flip_prob)
        self.brightness_max_delta = float(brightness_max_delta)
        self.contrast_lower = float(contrast_lower)
        self.contrast_upper = float(contrast_upper)
        self.gamma_range = None if gamma_range is None else tuple(gamma_range)
        self.clip_after_color = bool(clip_after_color)
        self.noise_std = float(noise_std)
        self.temporal_shift_max = int(temporal_shift_max)
        self.frame_drop_prob = float(frame_drop_prob)
        self.time_mask_prob = float(time_mask_prob)
        self.time_mask_max_frac = float(time_mask_max_frac)
        self.invert_prob = float(invert_prob)
        self.solarize_prob = float(solarize_prob)
        self.solarize_threshold = tuple(solarize_threshold)

    def call(self, x, training=None):
        if training is True:
            raise RuntimeError("This inference-only compatibility layer must not be used for training.")
        return tf.cast(x, tf.float32)

    def get_config(self):
        config = super().get_config()
        config.update({
            "image_size": self.image_size,
            "crop_scale": self.crop_scale,
            "rotation_max_deg": self.rotation_max_deg,
            "fill_mode": self.fill_mode,
            "fill_value": self.fill_value,
            "flip_prob": self.flip_prob,
            "brightness_max_delta": self.brightness_max_delta,
            "contrast_lower": self.contrast_lower,
            "contrast_upper": self.contrast_upper,
            "gamma_range": self.gamma_range,
            "clip_after_color": self.clip_after_color,
            "noise_std": self.noise_std,
            "temporal_shift_max": self.temporal_shift_max,
            "frame_drop_prob": self.frame_drop_prob,
            "time_mask_prob": self.time_mask_prob,
            "time_mask_max_frac": self.time_mask_max_frac,
            "invert_prob": self.invert_prob,
            "solarize_prob": self.solarize_prob,
            "solarize_threshold": self.solarize_threshold
        })
        return config


@register_keras_serializable(package="Custom")
class CrossAttention(layers.Layer):
    def __init__(self, units, **kwargs):
        super().__init__(**kwargs)
        self.units = int(units)
        self.weight_layer_v1 = layers.Dense(units, activation='sigmoid', use_bias=False, kernel_initializer=GlorotUniform()        )
        self.weight_layer_v2 = layers.Dense(units, activation='sigmoid', use_bias=False, kernel_initializer=GlorotUniform())

    def build(self, input_shape):
        shape_v1, shape_v2 = input_shape
        self.weight_layer_v1.build(shape_v2)  # weights for v1 are produced from v2
        self.weight_layer_v2.build(shape_v1)  # weights for v2 are produced from v1
        super().build(input_shape)
        
    def call(self, inputs, training=None):
        v1, v2 = inputs

        weights_v1 = self.weight_layer_v1(v2)
        weights_v2 = self.weight_layer_v2(v1)

        weighted_v1 = weights_v1 * v1
        weighted_v2 = weights_v2 * v2

        combined_output = weighted_v1 + weighted_v2

        return layers.Activation('sigmoid')(combined_output)

    def get_config(self):
        config = super().get_config()
        config.update({'units': self.units})
        return config


CUSTOM_OBJECTS = {
    "SequenceAugment": SequenceAugment,
    "seqaug>SequenceAugment": SequenceAugment,
    "CrossAttention": CrossAttention,
    "Custom>CrossAttention": CrossAttention
}


### 3.2 Lazy sequence datasets

No dataset is shuffled. Inference therefore remains aligned with the corresponding DataFrame row order.

Image data are decoded only when a batch is requested. This avoids materializing all 756 × 30 images in memory at once.


In [10]:
def parse_image(filename: tf.Tensor) -> tf.Tensor:
    image_string = tf.io.read_file(filename)
    image = tf.io.decode_jpeg(image_string, channels=IMAGE_SIZE[2])
    return image


def load_image_sequence(image_paths: tf.Tensor) -> tf.Tensor:
    images = tf.map_fn(parse_image, image_paths, fn_output_signature=tf.uint8)
    return tf.ensure_shape(images, (SEQUENCE_LENGTH, *IMAGE_SIZE))


In [11]:
def create_image_dataset(df: pd.DataFrame, batch_size=BATCH_SIZE) -> tf.data.Dataset:
    image_paths = np.array(df["image_paths"].tolist(), dtype=str)

    ds = tf.data.Dataset.from_tensor_slices(image_paths)
    ds = ds.map(load_image_sequence, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size)
    ds = ds.prefetch(1)

    return ds


def create_fea_dataset(df: pd.DataFrame, batch_size=BATCH_SIZE) -> tf.data.Dataset:
    fea_sequences = np.stack(df["fea_sequence"].to_numpy()).astype(np.float32)
    return tf.data.Dataset.from_tensor_slices(fea_sequences).batch(batch_size).prefetch(1)


def create_multimodal_dataset(df: pd.DataFrame, batch_size=BATCH_SIZE) -> tf.data.Dataset:
    fea_sequences = np.stack(df["fea_sequence"].to_numpy()).astype(np.float32)
    image_paths = np.array(df["image_paths"].tolist(), dtype=str)

    ds = tf.data.Dataset.from_tensor_slices((fea_sequences, image_paths))
    ds = ds.map(
        lambda fea, paths: {"fea_input": fea, "img_input": load_image_sequence(paths)},
        num_parallel_calls=tf.data.AUTOTUNE
    )
    ds = ds.batch(batch_size)
    ds = ds.prefetch(1)

    return ds


### 3.3 Prediction storage and validation

Dynamic model outputs are already in canonical class order. Each prediction is stored together with its full seven-class probability vector.


In [12]:
def add_predictions(df: pd.DataFrame, prefix: str, probabilities: np.ndarray) -> np.ndarray:
    probabilities = np.asarray(probabilities)

    assert probabilities.shape == (len(df), 7)
    assert np.all(probabilities >= 0)
    assert np.all(probabilities <= 1)
    assert np.allclose(probabilities.sum(axis=1), 1.0, atol=1e-4)

    predictions = np.argmax(probabilities, axis=1)

    df[f"{prefix}_pred_id"] = predictions
    df[f"{prefix}_pred"] = [CANONICAL_ID_TO_EMOTION[p] for p in predictions]

    for i, emotion in enumerate(EMOTIONS):
        df[f"{prefix}_prob_{emotion.lower()}"] = probabilities[:, i]

    return predictions


## 4. Dynamic image-sequence model

### 4.1 Load the frozen model and run inference

Only the image model and its current lazy dataset are kept in memory during this step.


In [13]:
image_model = keras.models.load_model(
    IMAGE_MODEL_PATH,
    custom_objects=CUSTOM_OBJECTS,
    compile=False,
    safe_mode=False
)

print("Input:", image_model.input_shape)
print("Output:", image_model.output_shape)

image_probabilities = image_model.predict(
    create_image_dataset(prediction_df, batch_size=BATCH_SIZE),
    verbose=1
)

image_predictions = add_predictions(prediction_df, "image", image_probabilities)


2026-09-17 17:18:00.094967: E tensorflow/core/util/util.cc:131] oneDNN supports DT_HALF only on platforms with AVX-512. Falling back to the default Eigen-based implementation if present.


Input: (None, 30, 224, 224, 3)
Output: (None, 7)
24/24 ━━━━━━━━━━━━━━━━━━━━ 95s 2s/step   


### 4.2 Verify the published dynamic image result

Expected result: **550 / 756 = 72.75%**.


In [14]:
image_correct = np.sum(image_predictions == prediction_df["true_label_id"].to_numpy())
image_accuracy = image_correct / len(prediction_df)

print(f"Dynamic Image: {image_correct}/756 = {image_accuracy:.6f}")

assert image_correct == 550


Dynamic Image: 550/756 = 0.727513


In [15]:
del image_model, image_probabilities
keras.backend.clear_session()
keras.mixed_precision.set_global_policy("mixed_float16")
gc.collect()


0

## 5. Dynamic FEA-sequence model

### 5.1 Load the frozen model and run inference

The FEA model is evaluated once on the 378 unique reenactment-level FEA sequences. Its predictions are then joined to both corresponding image-view rows.


In [16]:
fea_model = keras.models.load_model(FEA_MODEL_PATH, compile=False, safe_mode=False)

print("Input:", fea_model.input_shape)
print("Output:", fea_model.output_shape)

fea_probabilities = fea_model.predict(
    create_fea_dataset(fea_sequence_df, batch_size=BATCH_SIZE),
    verbose=1
)

fea_predictions = add_predictions(fea_sequence_df, "fea", fea_probabilities)


Input: (None, 30, 63)
Output: (None, 7)
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step  


### 5.2 Verify the published dynamic FEA result

Expected result: **296 / 378 = 78.31%**.


In [17]:
fea_correct = np.sum(fea_predictions == fea_sequence_df["fea_true_label_id"].to_numpy())
fea_accuracy = fea_correct / len(fea_sequence_df)

print(f"Dynamic FEA: {fea_correct}/378 = {fea_accuracy:.6f}")

assert fea_correct == 296


Dynamic FEA: 296/378 = 0.783069


In [18]:
fea_prediction_columns = [
    "reenactment_id",
    "fea_pred_id",
    "fea_pred",
    *[f"fea_prob_{emotion.lower()}" for emotion in EMOTIONS]
]

prediction_df = prediction_df.merge(
    fea_sequence_df[fea_prediction_columns],
    on="reenactment_id",
    how="left",
    validate="many_to_one"
)

assert prediction_df["fea_pred_id"].notna().all()
assert (prediction_df.groupby("reenactment_id")["fea_pred_id"].nunique() == 1).all()

fea_correct_paired = np.sum(prediction_df["fea_pred_id"].to_numpy() == prediction_df["true_label_id"].to_numpy())
assert fea_correct_paired == 592

print(f"FEA paired representation: {fea_correct_paired}/756 = {fea_correct_paired / 756:.6f}")


FEA paired representation: 592/756 = 0.783069


In [19]:
del fea_model, fea_probabilities
keras.backend.clear_session()
keras.mixed_precision.set_global_policy("mixed_float16")
gc.collect()


0

## 6. Dynamic multimodal model

### 6.1 Load the frozen intermediate-fusion model

The model expects one 30 × 63 FEA sequence and one synchronized 30-frame image sequence. The dataset passes these inputs by their saved input names.


In [20]:
print("Mixed precision policy:", keras.mixed_precision.global_policy())
assert keras.mixed_precision.global_policy().name == "mixed_float16"

multimodal_model = keras.models.load_model(
    MULTIMODAL_MODEL_PATH,
    custom_objects=CUSTOM_OBJECTS,
    compile=False,
    safe_mode=False
)

print("Inputs:", multimodal_model.input_shape)
print("Output:", multimodal_model.output_shape)

input_names = [tensor.name.split(":")[0] for tensor in multimodal_model.inputs]
print("Input names:", input_names)

assert set(input_names) == {"fea_input", "img_input"}


Mixed precision policy: <DTypePolicy "mixed_float16">
Inputs: [(None, 30, 63), (None, 30, 224, 224, 3)]
Output: (None, 7)
Input names: ['fea_input', 'img_input']


### 6.2 Run inference and verify the published multimodal result

Expected result: **617 / 756 = 81.61%**.


In [21]:
multimodal_probabilities = multimodal_model.predict(
    create_multimodal_dataset(prediction_df, batch_size=BATCH_SIZE),
    verbose=1
)

multimodal_predictions = add_predictions(prediction_df, "multimodal", multimodal_probabilities)

multimodal_correct = np.sum(multimodal_predictions == prediction_df["true_label_id"].to_numpy())
multimodal_accuracy = multimodal_correct / len(prediction_df)

print(f"Dynamic Multimodal: {multimodal_correct}/756 = {multimodal_accuracy:.6f}")

assert multimodal_correct == 617


24/24 ━━━━━━━━━━━━━━━━━━━━ 103s 2s/step  
Dynamic Multimodal: 617/756 = 0.816138


In [22]:
del multimodal_model, multimodal_probabilities
keras.backend.clear_session()
gc.collect()


0

## 7. Final validation and export

### 7.1 Integrity checks

These checks verify the expected number of view-level samples and reenactments, the balanced test-set class distribution, complete model predictions, and identical FEA predictions across the two views belonging to each reenactment.


In [23]:
assert len(prediction_df) == 756
assert prediction_df["sample_id"].is_unique
assert prediction_df["reenactment_id"].nunique() == 378

assert (prediction_df.groupby("reenactment_id").size() == 2).all()
assert (prediction_df.groupby("reenactment_id")["camera_index"].nunique() == 2).all()
assert (prediction_df.groupby("true_label_id").size() == 108).all()

assert prediction_df[["image_pred_id", "fea_pred_id", "multimodal_pred_id"]].notna().all().all()
assert (prediction_df.groupby("reenactment_id")["fea_pred_id"].nunique() == 1).all()

for prefix in ["image", "fea", "multimodal"]:
    probability_columns = [f"{prefix}_prob_{emotion.lower()}" for emotion in EMOTIONS]
    assert prediction_df[probability_columns].notna().all().all()
    assert np.allclose(prediction_df[probability_columns].sum(axis=1), 1.0, atol=1e-4)

print("Final prediction table passed all integrity checks.")


Final prediction table passed all integrity checks.


### 7.2 Summarize reproduced results

The FEA accuracy is shown on the 378 unique reenactments. Image and Multimodal accuracy are shown on the 756 view-level samples.


In [24]:
image_correct = (prediction_df["image_pred_id"] == prediction_df["true_label_id"]).sum()
multimodal_correct = (prediction_df["multimodal_pred_id"] == prediction_df["true_label_id"]).sum()

unique_fea_df = prediction_df.drop_duplicates("reenactment_id")
fea_correct = (unique_fea_df["fea_pred_id"] == unique_fea_df["true_label_id"]).sum()

print(f"{'image':<12}: {image_correct:>3}/756 = {image_correct / 756:.4%}")
print(f"{'fea':<12}: {fea_correct:>3}/378 = {fea_correct / 378:.4%}")
print(f"{'multimodal':<12}: {multimodal_correct:>3}/756 = {multimodal_correct / 756:.4%}")


image       : 550/756 = 72.7513%
fea         : 296/378 = 78.3069%
multimodal  : 617/756 = 81.6138%


### 7.3 Optional static-key alignment check

If `static_test_predictions.csv` is available in the current working directory, this check confirms that the static and dynamic exports use exactly the same 756 `sample_id` values and 378 `reenactment_id` values. This is useful for the later Static-vs.-Dynamic significance tests.


In [25]:
static_predictions_path = Path("static_test_predictions.csv")

if static_predictions_path.exists():
    static_df = pd.read_csv(static_predictions_path)

    assert set(static_df["sample_id"]) == set(prediction_df["sample_id"])
    assert set(static_df["reenactment_id"]) == set(prediction_df["reenactment_id"])

    print("Static and dynamic prediction keys align exactly.")
else:
    print("static_test_predictions.csv not found; optional key-alignment check skipped.")


Static and dynamic prediction keys align exactly.


### 7.4 Export per-sample predictions

The exported CSV contains one row per dynamic image-view sequence. Raw 30-frame image-path lists and raw 30 × 63 FEA sequences are omitted; identifiers, sequence-directory paths, predictions, and class probabilities are retained.


In [26]:
output_columns = [
    "sample_id",
    "reenactment_id",
    "timestamp",
    "set_id",
    "participant_id",
    "level_id",
    "emoji_id",
    "camera_index",
    "perspective",
    "true_label_id",
    "true_label",
    "image_sequence_path"
]

prediction_columns = [
    c for c in prediction_df.columns
    if c.startswith("image_") or c.startswith("fea_") or c.startswith("multimodal_")
]

prediction_columns = [
    c for c in prediction_columns
    if c not in {"image_sequence_path", "image_paths", "fea_timestamps", "fea_sequence", "fea_true_label_id"}
]

output_df = prediction_df[output_columns + prediction_columns].copy()
output_df.to_csv("dynamic_test_predictions.csv", index=False)

print(f"Exported {len(output_df)} rows to dynamic_test_predictions.csv")
display(output_df.head())


Exported 756 rows to dynamic_test_predictions.csv


,sample_id,reenactment_id,timestamp,set_id,participant_id,level_id,emoji_id,camera_index,perspective,true_label_id,...,fea_prob_surprise,multimodal_pred_id,multimodal_pred,multimodal_prob_anger,multimodal_prob_disgust,multimodal_prob_fear,multimodal_prob_happiness,multimodal_prob_neutral,multimodal_prob_sadness,multimodal_prob_surprise
0,1700478995850-2-1-1-0-0-0,1700478995850-2-1-1-0-0,1700478995850,2,1,1,0,0,Central,0,...,0.020970,0,Anger,0.798098,0.054739,0.005278,0.006042,0.009932,0.117587,0.008324
1,1700478995850-2-1-1-0-0-1,1700478995850-2-1-1-0-0,1700478995850,2,1,1,0,1,Side,0,...,0.020970,0,Anger,0.586167,0.127297,0.010622,0.006152,0.022692,0.239523,0.007547
2,1700478998549-2-1-1-1-5-0,1700478998549-2-1-1-1-5,1700478998549,2,1,1,1,0,Central,5,...,0.000122,5,Sadness,0.016794,0.003465,0.003547,0.006520,0.007196,0.958934,0.003545
3,1700478998549-2-1-1-1-5-1,1700478998549-2-1-1-1-5,1700478998549,2,1,1,1,1,Side,5,...,0.000122,5,Sadness,0.016804,0.003480,0.003540,0.006520,0.007197,0.958908,0.003550
4,1700479001137-2-1-1-2-3-0,1700479001137-2-1-1-2-3,1700479001137,2,1,1,2,0,Central,3,...,0.000419,3,Happiness,0.002462,0.004623,0.003282,0.982725,0.001283,0.002237,0.003387
